## **Lab 4: PySpark**


### **1a. Installation check**


In [1]:
import findspark
findspark.init()

from operator import add
from pyspark import SparkConf, SparkContext

conf = (SparkConf().setMaster("local[2]").setAppName("Lab4")
        .set("spark.hadoop.fs.defaultFS", "file:///"))  # Read local files, not HDFS.
sc = SparkContext.getOrCreate(conf)
sc.setLogLevel("ERROR")
print("Spark:", sc.version, "| Test sum:", sc.parallelize([1, 2, 3]).sum())


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/08 15:06:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark: 4.2.0 | Test sum: 6


### **1b. Word count**


In [2]:
word_counts = (sc.textFile("data/wordcount.txt").flatMap(str.split)
               .map(lambda word: (word, 1)).reduceByKey(add)
               .sortByKey().collect())
print(word_counts)


[('hadoop', 1), ('hello', 2), ('spark', 3)]


### **1c. MovieLens: rating distribution for each movie**

In [3]:
ratings = sc.textFile("data/ml-100k/u.data").map(str.split)
rating_counts = (ratings.map(lambda r: ((int(r[1]), int(r[2])), 1))
                 .reduceByKey(add))
movie_counts = (rating_counts.map(lambda x: (x[0][0], (x[0][1], x[1])))
                .groupByKey().mapValues(dict).sortByKey().collect())
movie_distribution = [(movie, [counts.get(rating, 0) for rating in range(1, 6)])
                      for movie, counts in movie_counts]
print(movie_distribution[:10])


[(1, [8, 27, 96, 202, 119]), (2, [8, 17, 55, 42, 9]), (3, [11, 20, 25, 23, 11]), (4, [6, 24, 57, 93, 29]), (5, [4, 11, 32, 33, 6]), (6, [2, 4, 5, 7, 8]), (7, [9, 31, 89, 164, 99]), (8, [4, 13, 47, 71, 84]), (9, [11, 18, 59, 114, 97]), (10, [2, 7, 21, 33, 26])]


### **2. Average friends by age**


In [4]:
friends = sc.textFile("data/friends_test.csv").map(lambda line: line.split(","))
average_friends = (friends.map(lambda r: (int(r[2]), int(r[3])))
                   .mapValues(lambda count: (count, 1))
                   .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
                   .mapValues(lambda total: total[0] / total[1])
                   .sortByKey().collect())
for age, average in average_friends:
    print(age, round(average, 2))


18 343.38
19 213.27
20 165.0
21 350.88
22 206.43
23 246.3
24 233.8
25 197.45
26 242.06
27 228.12
28 209.1
29 215.92
30 235.82
31 267.25
32 207.91
33 325.33
34 245.5
35 211.62
36 246.6
37 249.33
38 193.53
39 169.29
40 250.82
41 268.56
42 303.5
43 230.57
44 282.17
45 309.54
46 223.69
47 233.22
48 281.4
49 184.67
50 254.6
51 302.14
52 340.64
53 222.86
54 278.08
55 295.54
56 306.67
57 258.83
58 116.55
59 220.0
60 202.71
61 256.22
62 220.77
63 384.0
64 281.33
65 298.2
66 276.44
67 214.62
68 269.6
69 235.2


### **3 and 4. Temperature extrema**


In [5]:
lines = sc.textFile("data/temp.csv")
header = lines.first()
temperatures = (lines.filter(lambda line: line != header)
                .map(lambda line: line.split(","))
                .map(lambda r: (r[0], r[1], r[2], int(r[3]))))

def temperature_extrema(kind, operation):
    selected = temperatures.filter(lambda r: r[2] == kind)
    overall = selected.map(lambda r: r[3]).reduce(operation)
    by_item = selected.map(lambda r: (r[0], r[3])).reduceByKey(operation).sortByKey().collect()
    by_station = selected.map(lambda r: (r[1], r[3])).reduceByKey(operation).sortByKey().collect()
    print("Overall:", overall)
    print("By item:", by_item)
    print("By station:", by_station)
    return overall, by_item, by_station


### **3. TMIN: minimum temperatures**


In [6]:
minimum_temperatures = temperature_extrema("TMIN", min)


Overall: -148
By item: [('EZE00100082', -135), ('ITE00100554', -148)]
By station: [('18000101', -148), ('18000102', -130), ('18000103', -73), ('18000104', -74), ('18000105', -58), ('18000106', -57), ('18000107', -50), ('18000108', -31), ('18000109', -46), ('18000110', -75), ('18000111', -62), ('18000112', -60), ('18000113', -60), ('18000114', -35), ('18000115', -23), ('18000116', -37), ('18000117', -35), ('18000118', 9), ('18000119', 34), ('18000120', 17), ('18000121', 8), ('18000122', -16), ('18000123', -16), ('18000124', -3), ('18000125', 2), ('18000126', 16), ('18000127', 15), ('18000128', 33), ('18000129', 19), ('18000130', 3), ('18000131', 23), ('18000201', 17), ('18000202', 19), ('18000203', 2), ('18000204', 39), ('18000205', 22), ('18000206', -3), ('18000207', -22), ('18000208', -39), ('18000209', -36), ('18000210', -69), ('18000211', -102), ('18000212', -78), ('18000213', -42), ('18000214', -35), ('18000215', 0), ('18000216', -6), ('18000217', -13), ('18000218', -52), ('1800021

### **4. TMAX: maximum temperatures**


In [7]:
maximum_temperatures = temperature_extrema("TMAX", max)


Overall: 323
By item: [('EZE00100082', 323), ('ITE00100554', 323)]
By station: [('18000101', -75), ('18000102', -44), ('18000103', -10), ('18000104', 0), ('18000105', 10), ('18000106', 13), ('18000107', 31), ('18000108', 29), ('18000109', 35), ('18000110', 46), ('18000111', 66), ('18000112', 41), ('18000113', 23), ('18000114', 41), ('18000115', 54), ('18000116', 56), ('18000117', 84), ('18000118', 59), ('18000119', 66), ('18000120', 60), ('18000121', 80), ('18000122', 81), ('18000123', 85), ('18000124', 85), ('18000125', 98), ('18000126', 75), ('18000127', 73), ('18000128', 79), ('18000129', 73), ('18000130', 66), ('18000131', 85), ('18000201', 91), ('18000202', 67), ('18000203', 56), ('18000204', 104), ('18000205', 79), ('18000206', 75), ('18000207', 73), ('18000208', 50), ('18000209', 35), ('18000210', 38), ('18000211', 21), ('18000212', 13), ('18000213', 13), ('18000214', 13), ('18000215', 21), ('18000216', 75), ('18000217', 71), ('18000218', 48), ('18000219', 50), ('18000220', 66),

In [8]:
sc.stop()
